# Imports

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import time
import tensorflow as tf
from tensorflow import keras
from keras import layers
import sklearn
import glob
import json
from sklearn.metrics import roc_curve,auc
import os
import matplotlib.gridspec as GS
from keras.optimizers import Adam
from keras.saving import register_keras_serializable

2026-06-11 23:15:24.739568: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-11 23:15:26.179402: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781212526.656290     941 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781212526.833275     941 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-11 23:15:27.730271: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [4]:
with open("../clipped_models/clipped_vae_3layer_latent3_lastfilter4_beta1e-2/clipped_vae_3layer_latent3_lastfilter4_beta1e-2_all_roc_data.json",'r') as f:
    data = json.load(f)    

In [11]:
data['targetMSE']['thresholds'].keys()

dict_keys(['w', 'z', 't', 'combined_signal'])

# Training Functions

We first define the functions to train our VAE and calculate the ROC curve data for each combination of signal and score.

In [2]:
def recursive_dict_conversion_tolist(d):
    """
    Recursively converts values of a specific type within a dictionary to a target type.

    Args:
        d (dict): The dictionary to modify.
        target_type (type): The desired type to c onvert values to.

    Returns:
        dict: The modified dictionary with values converted.
    """
    new_dict = {}
    for k, v in d.items():
        if isinstance(v, dict):
            new_dict[k] = recursive_dict_conversion_tolist(v)
        elif isinstance(v, np.ndarray): 
            new_dict[k] = v.tolist()
        else:
            new_dict[k] = v

    return new_dict

def get_roc_data_dict_all_targets(model,name,background,signal_dict,beta):
    """
    Uses the [model] to make predictions on the [background] and [signal_dict] data.
    Then, produces a dictionary that contains the FPRs, TPRs, and thresholds for each
    signal evaluated against the background for each of the five anomaly scores given
    as Equations (2)-(6) in the paper. The dictionary also saves the scores for the
    background and signals.
    
    Creates and saves two plots:
        1. ROC curves for each score on the combined signal
        2. ROC curves for each score with separate currves for each signal

    The dictionary is of the following format (when the signal dict contains keys and data for 'w','z','t')
        roc_data_dict = {'targetMSE' : {
                                        'fprs' :
                                            {'w' : [..] , 'z' : [...], 't' : [...], 'combined_signal' : [...]}, 
                                        tprs' :
                                            {'w' : [..] , 'z' : [...], 't' : [...], 'combined_signal' : [...]},
                                        'thresholds' :
                                            {'w' : [..] , 'z' : [...], 't' : [...], 'combined_signal' : [...]},
                                        'scores' :
                                            {'background','w' : [..] , 'z' : [...], 't' : [...], 'combined_signal' : [...]},
                                        },
                         'targetMSE_KL' : "
                         'targetKL' : "
                         'targetmus' : "
                         'targetsigmas : "
                        }
    Where target* refers to each score D_*.

    Args:
        - model (loaded .keras model) : trained keras VAE model
        - name (str) : name to use to save the data and create the plots
        - background (np.array) : numpy array of shape [N,24,24,1] that contains the quark and gluon training
                      samples
        - signal_dict (dict of str : np.array) : dictionary that contains keys labelling the signals and values of numpy arrays
                       of the data for each signal of shape [N,24,24,1]. Should contain data for keys: 'w','z','t'
        - beta (float) : the beta hyperparameter used to balance the MSE and KL terms in the D_MSE+KL score.
                Should match the value used during training of the model


    Returns:
        The roc_data_dict described above.
    """
    print("Model name: ",name)
    roc_data_dict = {}

    # Make predictions on background data
    background_preds = model.predict(background,verbose=0)

    # Calculate the background scores
    background_MSE = np.mean(np.square(background_preds['output'] - background),axis=(1,2,3))
    
    mu = background_preds['latent_z_mean']
    log_sigma = background_preds['latent_z_log_var']
    
    mus = np.sum(np.square(mu),axis=-1)
    sigmas = np.mean((np.sqrt(np.exp(log_sigma)) - 1),axis=-1)
    KL = 0.5 * (np.sum(np.square(mu) + np.exp(log_sigma) - 1 - log_sigma,axis=-1))
    MSE_KL = (1-beta)*background_MSE + (beta)*KL

    target_fpr=0.01

    # Iterate over the scores
    fig,axs = plt.subplots(1,5,figsize=(16,3),layout='constrained')
    for i,target in enumerate(['MSE','mus','sigmas','KL','MSE_KL']):
        fprs = {}
        tprs = {}
        thresholds = {}
        scores_dict = {}

        # Iterate over the signals
        for j,key in enumerate(signal_dict.keys()):
            signal = signal_dict[key]

            # Calcualte the signal scores
            signal_preds = model.predict(signal,verbose=0)
            signal_MSE = np.mean(np.square(signal_preds['output'] - signal),axis=(1,2,3))

            mu_signal = signal_preds['latent_z_mean']
            log_sigma_signal = signal_preds['latent_z_log_var']
        
            KL_signal = 0.5 * (np.sum(np.square(mu_signal) + np.exp(log_sigma_signal) - 1 - log_sigma_signal,axis=-1))
            MSE_KL_signal = (1-beta)*signal_MSE + (beta)*KL_signal

            if target=='MSE':
                background_target = background_MSE
                signal_target = signal_MSE
            elif target=='mus':
                background_target = mus
                signal_target = np.sum(np.square(mu_signal),axis=-1)
            elif target == 'sigmas':
                background_target = sigmas
                signal_target = np.mean((np.sqrt(np.exp(log_sigma_signal)) - 1),axis=-1)
            elif target == 'KL':
                background_target = KL
                signal_target = KL_signal
            elif target == 'MSE_KL':
                background_target = MSE_KL
                signal_target = MSE_KL_signal
            else:
                raise "Invalid target"

            # Calculate the ROC curves
            labels = np.concatenate((np.full(shape=signal_target.shape[0],fill_value=1),np.full(shape=background_target.shape[0],fill_value=0)))
            scores = np.concatenate((signal_target,background_target))
            fpr,tpr,threshold = roc_curve(y_true = labels,y_score=scores)

            fprs[key] = fpr
            tprs[key] = tpr
            thresholds[key] = threshold
            scores_dict['background'] = background_target
            scores_dict[key] = signal_target

            idx = np.argmin(np.abs(fpr - target_fpr))
            tpr_at_target = tpr[idx]
            label = f"{key}"

            # Plot the ROC curves
            axs[i].plot(fpr,tpr,label=label)
        axs[i].plot([0,1],[0,1],label='random guess',color='grey',linestyle='dashed')
        axs[i].set_title(target)
        axs[i].set_xlabel("FPR")
        axs[i].set_ylabel("TPR")
        axs[i].set_xscale('log')
        axs[i].set_yscale('log')
        roc_data_dict[f'target{target}'] = {'fprs' : fprs, "tprs" : tprs, "thresholds":thresholds,'scores':scores_dict}

        # Calculate the ROC curves on the combined signal data
        background_scores = background_target
        combined_signals = np.concatenate([scores_dict[signal] for signal in ['w','z','t']])
        
        combined_labels = np.concatenate((np.full(shape=combined_signals.shape[0],fill_value=1),np.full(shape=background_scores.shape[0],fill_value=0)))
        combined_scores = np.concatenate((combined_signals,background_scores))
        combined_fpr,combined_tpr,combined_threshold = roc_curve(y_true = combined_labels,y_score=combined_scores)
        fprs['combined_signal'] = combined_fpr
        tprs['combined_signal'] = combined_tpr
        thresholds['combined_signal'] = combined_threshold
        scores_dict['combined_signal']= combined_signals        
        roc_data_dict[f'target{target}'] = {'fprs' : fprs, "tprs" : tprs, "thresholds":thresholds,'scores':scores_dict}

    
    axs[4].legend(bbox_to_anchor=(1.05,0.95))
    plt.suptitle(name)
    plt.savefig(f"{name}_all_ROC.png")
    plt.show()

    # Create a second plot with just the combined signals
    plt.figure(figsize=(8,6))
    for k,target in enumerate(roc_data_dict.keys()):
        combined_fpr = roc_data_dict[target]['fprs']['combined_signal']
        combined_tpr = roc_data_dict[target]['tprs']['combined_signal']
        plt.plot(combined_fpr,combined_tpr,label=target,color=f"C{k}")
    plt.plot([0,1],[0,1],label='random guess',color='grey',linestyle='dashed')
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('FPR')
    plt.ylabel('TPR')
    plt.legend()
    plt.title(f"{name} \n combined signals")
    plt.savefig(f"{name}_combined_ROC.png")
    plt.show()
    
    return roc_data_dict

def plot_results(title,model_history,input_data,reconstructed_data):
    '''
    Creates the following subplots:
        1. The training loss
        2. The validation loss
        3. The separated training loss components
        4. A grid of original image and the model predictions ("reconstructions")
           for 3 gluon events and 2 quark events

    Args:
        - title (str) : title for the plot
        - model_history: the history object returned from tf.callbacks.History when calling
                         keras model.fit()
        - input_data (np.array) : test data numpy array of shape [N,24,24,1] with gluon events in
                                  the first half and quark events in the second half
        - reconstructed_data (np.array) : model predictions on input_data

    Returns:
        - Matplotlib figure
    '''
    fig = plt.figure(figsize=(18,8))
    
    gs = GS.GridSpec(nrows=3, ncols=6,height_ratios=[1,1,1])

    # Create the loss plots
    if model_history != []:
        ax1 = fig.add_subplot(gs[0,0:2])
        ax1.plot(np.array(model_history.history['loss']))
        ax1.set_title('Loss')
        ax1.set_xlabel("Epoch")
        ax1.set_ylabel("Loss")
    
        ax2 = fig.add_subplot(gs[0,2:4])
        ax2.plot(np.array(model_history.history['val_loss']))
        ax2.set_title('Val Loss')
        ax2.set_xlabel("Epoch")
        ax2.set_ylabel("Loss")

        
        ax3 = fig.add_subplot(gs[0,4:6])
        ax3.set_title('Losses')
        ax3.set_xlabel("Epoch")
        ax3.set_ylabel("Loss")
        ax3.plot(model_history.history['loss'],label='loss')
        ax3.plot(model_history.history['output_loss'],label='output_loss')
        ax3.plot(model_history.history['latent_z_mean_loss'],label='latent_mean_loss')
        ax3.plot(model_history.history['latent_z_log_var_loss'],label='latent_log_var_loss')
        ax3.set_yscale('log')
        ax3.legend()

    # Create the original and reconstructed image plots
    labels = ["j_g","j_g","j_g","j_q","j_q","j_q"]
    events = [0,1,2,-1,-2,-3]
    event_label = ["0","1","2","0","1","2"]
    for i in range(6):
        event_num = events[i]
        vmin= min(np.min(input_data[event_num]),np.min(reconstructed_data[event_num]))
        vmax= max(np.max(input_data[event_num]),np.max(reconstructed_data[event_num]))
        ax3 = fig.add_subplot(gs[1, i],aspect='equal')
        ax3.set_title(labels[i] + " event " + event_label[i])
        im3 = ax3.imshow(input_data[event_num],vmin=vmin,vmax=vmax,interpolation="none")
        plt.colorbar(im3,ax=ax3,shrink=0.8)
        
        ax4 = fig.add_subplot(gs[2, i],aspect='equal')
        ax4.set_title(labels[i] + " event " + event_label[i])    
        im4 = ax4.imshow(reconstructed_data[event_num],vmin=vmin,vmax=vmax,interpolation="none")
        plt.colorbar(im4,ax=ax4,shrink=0.8)

        if i==0:
            ax3.set_ylabel("Original")
            ax4.set_ylabel("Reconstruction")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()
    return fig

def train_model(directory_name,save_name, model, train_data, test_data, val_data, signals_dict, beta, epochs=300):
    '''
    Trains keras [model] using [train_data] as the training dataset, [val_data] as the validation dataset, and [test_data]
    for the plots. Computes the ROC curve data ("roc_data_dict") for each signal and each target score by calling
    get_roc_data_dict_all_targets. Saves the trained model as a .keras object and the loss and reconstruction plots to
    the directory called [directory_name].

    Args:
        - directory_name (str) : directory to save the trained model and plots
        - model : keras model to train
        - train_data (tf.data.Dataset) : train dataset
        - val_data (tf.data.Dataset) : val dataset
        - signals_dict (dict of key (str) : value (np.array) pairs with data for each signal)
        - beta : the beta hyperparameter value to train with
        - epochs : the default number of epochs to train

    Returns:
        model_history, roc_data_dict        
    
    '''
    # Create directory
    os.makedirs(directory_name) 
    save_name = os.path.join(directory_name,save_name)

    # Train the model with early stopping and ReduceLROnPlateau
    print("Starting training")
    t_1 = time.time()
    callback = keras.callbacks.EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)
    reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.1,patience=5, min_lr=1e-6)
    model_history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=epochs,
        callbacks=[callback,reduce_lr]
    )
    
    t_2 = time.time()
    print('Training took {} minutes'.format((t_2-t_1)/60))

    # Plot the losses
    plt.plot(model_history.history['loss'],label='loss')
    plt.plot(model_history.history['val_loss'],label='val_loss')
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Loss")
    plt.legend()
    plt.savefig(f"{save_name}_loss.png")
    plt.show()
    
    # Save the models
    print("Saving model as:", save_name)
    model.save(save_name + ".keras")

    print("Making predictions on test data")
    preds = model.predict(test_data)['output']
    
    print("Creating plots")

    # Plot the losses, and the original and reconstructed images
    imgs_plots = plot_results(save_name,model_history,test_data,preds)
    imgs_plots.savefig(save_name,facecolor="white")

    # Calculate the ROC data and save the dictionary as a json file
    roc_data_all_targets = get_roc_data_dict_all_targets(model,save_name,test_data,signals_dict,beta)
    roc_data_all_targets_lists = recursive_dict_conversion_tolist(roc_data_all_targets)
    with open(f"{save_name}_all_roc_data.json", "w") as f:
        json.dump(roc_data_all_targets_lists, f, indent=4) # indent for pretty printing
    
    return model_history,roc_data_all_targets

# Model Functions

We now define our VAE architectures that we will test.

In [3]:
@register_keras_serializable()
def sampling(args):
    '''
    Samples from a Gaussian distribution defined by mu=ars[0] and log(sigma^2)=args[1]

    Args:
        args: (mu, log(sigma^2))
    '''
    z_mean, z_log_var = args
    epsilon = tf.random.normal(shape=tf.shape(z_mean))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# KL losses
# We break the KL portion of the loss into two parts: one part that uses the mu latent space
# variable and the other that uses the sigma latent space variable.
@register_keras_serializable()
def kl_mu_loss(y_true,y_pred):
    '''
    Args:
        - y_true: dummy variable to match the expected loss function signature
        - y_pred: the mu latent space variable
        
    Returns the terms in the KL loss that use mu
    
    '''
    return 0.5*tf.reduce_sum(tf.square(y_pred))

@register_keras_serializable()
def kl_sigma_loss(y_true,y_pred):
    '''
    Args:
        - y_true : dummy variable to match the expected loss function signature
        - y_pred : the log(sigma^2) latent space variable
        
    Returns the terms in the KL loss that use sigma
    '''
    return 0.5*tf.reduce_sum(tf.exp(y_pred) - 1. - y_pred)

def get_3layer_vae(latent_dim,num_filters,last_filter=8):
    '''
    Args:
        - latent_dim (int) : the dimension of the latent space
        - num_filters (int) : the number of filters to use in the first two encoder
                              and last two decoder layers
        - last_filter (int) : the number of filters to use in the last encoder and
                              first decoder layers
                              
    Returns a CNN-VAE with 3 layers in the encoder and 3 layers in the decoder.
    '''
    name = "vae_3layer_latent" + str(latent_dim) + "_lastfilter" + str(last_filter)
    encoder_input = keras.Input(shape=(24,24,1))
    x = keras.layers.Conv2D(num_filters,(4,4),2,activation="relu")(encoder_input)
    x = keras.layers.Conv2D(num_filters,(3,3),2,activation="relu")(x)
    x = keras.layers.Conv2D(last_filter,(2,2),1,activation="relu")(x)
    x = keras.layers.Flatten()(x)
    z_mean = keras.layers.Dense(latent_dim, name="latent_z_mean")(x)
    z_log_var = layers.Dense(latent_dim, name = "latent_z_log_var")(x)
    z = keras.layers.Lambda(sampling, output_shape=(latent_dim,), name='latent_samp')([z_mean, z_log_var])

    x = keras.layers.Dense(4*4*last_filter,activation="relu")(z)
    x = keras.layers.Reshape((4,4,last_filter))(x)
    x = keras.layers.Conv2DTranspose(last_filter,(2,2),1,activation="relu")(x)
    x = keras.layers.Conv2DTranspose(num_filters,(3,3),2,activation="relu")(x)
    decoder_outputs = keras.layers.Conv2DTranspose(1,(4,4),2,activation="linear",name="output")(x)

    vae = keras.Model(inputs=encoder_input,
                       outputs={"output": decoder_outputs,
                                "latent_z_mean": z_mean,
                                "latent_z_log_var": z_log_var},
                       name="vae_3layer")

    print(name)
    return vae,name


def get_3layer_vae_v2(latent_dim,num_filters,last_filter=8):
    '''
    Args:
        - latent_dim (int) : the dimension of the latent space
        - num_filters (int) : the number of filters to use in the first two encoder
                              and last two decoder layers
        - last_filter (int) : the number of filters to use in the last encoder and
                              first decoder layers
                              
    Returns a CNN-VAE with 3 layers in the encoder and 3 layers in the decoder
    and a name for the model. Has different convolutional kernel shapes than
    the previous function.
    '''
    name = "vae_3layer_v2_latent" + str(latent_dim) + "_lastfilter" + str(last_filter)
    encoder_input = keras.Input(shape=(24,24,1))
    x = keras.layers.Conv2D(num_filters,(4,4),2,activation="relu")(encoder_input)
    x = keras.layers.Conv2D(num_filters,(3,3),2,activation="relu")(x)
    x = keras.layers.Conv2D(last_filter,(3,3),2,activation="relu")(x)
    x = keras.layers.Flatten()(x)
    z_mean = keras.layers.Dense(latent_dim, name="latent_z_mean")(x)
    z_log_var = layers.Dense(latent_dim, name = "latent_z_log_var")(x)
    z = keras.layers.Lambda(sampling, output_shape=(latent_dim,), name='latent_samp')([z_mean, z_log_var])
    x = keras.layers.Dense(2*2*last_filter,activation="relu")(z)
    x = keras.layers.Reshape((2,2,last_filter))(x)
    x = keras.layers.Conv2DTranspose(last_filter,(3,3),2,activation="relu")(x)
    x = keras.layers.Conv2DTranspose(num_filters,(3,3),2,activation="relu")(x)
    decoder_outputs = keras.layers.Conv2DTranspose(1,(4,4),2,activation="linear",name="output")(x)

    vae = keras.Model(inputs=encoder_input,
                       outputs={"output": decoder_outputs,
                                "latent_z_mean": z_mean,
                                "latent_z_log_var": z_log_var},
                       name="vae_3layer_v2")
    print(name)
    return vae,name

def get_4layer_vae(latent_dim,num_filters,small_latent=True,last_filter=8):
    '''
    Args:
        - latent_dim (int) : the dimension of the latent space
        - num_filters (int) : the number of filters to use in the first three encoder
                              and last three decoder layers
        - last_filter (int) : the number of filters to use in the last encoder and
                              first decoder layers
        - small_latent (int) : option to use a smaller latent space by increasing
                               the stride of the last encoder and first decoder layers
                              
    Returns a CNN-VAE with 4 layers in the encoder and 4 layers in the decoder
    and a name for the model.
    '''
    encoder_input = keras.Input(shape=(24,24,1))
    x = keras.layers.Conv2D(num_filters,(3,3),1,activation="relu")(encoder_input)
    x = keras.layers.Conv2D(num_filters,(4,4),2,activation="relu")(x)
    x = keras.layers.Conv2D(num_filters,(3,3),1,activation="relu")(x)
    if small_latent:
        x = keras.layers.Conv2D(last_filter,(2,2),2,activation="relu")(x)
        latent_num = 4*4*last_filter
        name = "vae_4layer_small_latent" + str(latent_dim) + "_lastfilter" + str(last_filter)

    else:
        x = keras.layers.Conv2D(last_filter,(2,2),1,activation="relu")(x)
        latent_num = 7*7*last_filter
        name = "vae_4layer_large_latent" + str(latent_dim) + "_lastfilter" + str(last_filter)
    x = keras.layers.Flatten()(x)
    z_mean = keras.layers.Dense(latent_dim, name="latent_z_mean")(x)
    z_log_var = layers.Dense(latent_dim, name = "latent_z_log_var")(x)
    z = keras.layers.Lambda(sampling, output_shape=(latent_dim,), name='latent_samp')([z_mean, z_log_var])
    x = keras.layers.Dense(latent_num,activation="relu")(z)
    if small_latent:
        x = keras.layers.Reshape((4,4,last_filter))(x)
        x = keras.layers.Conv2DTranspose(last_filter,(2,2),2,activation="relu")(x)
    else:
        x = keras.layers.Reshape((7,7,last_filter))(x)
        x = keras.layers.Conv2DTranspose(last_filter,(2,2),1,activation="relu")(x)
    x = keras.layers.Conv2DTranspose(num_filters,(3,3),1,activation="relu")(x)
    x = keras.layers.Conv2DTranspose(num_filters,(4,4),2,activation="relu")(x)
    decoder_outputs = keras.layers.Conv2DTranspose(1,(3,3),1,activation="linear",name="output")(x) # QUESTION: Can I use RELU here? I think so because everything is non-negative

    vae = keras.Model(inputs=encoder_input,
                       outputs={"output": decoder_outputs,
                                "latent_z_mean": z_mean,
                                "latent_z_log_var": z_log_var},
                       name=name)
    print(name)
    return vae,name

def get_4layer_vae_v2(latent_dim,num_filters,last_filter=8):
    '''
    Args:
        - latent_dim (int) : the dimension of the latent space
        - num_filters (int) : the number of filters to use in the first three encoder
                              and last three decoder layers
        - last_filter (int) : the number of filters to use in the last encoder and
                              first decoder layers
                              
    Returns a CNN-VAE with 4 layers in the encoder and 4 layers in the decoder and
    a name for the model. Uses different kernel sizes than get_4layer_vae.
    '''
    name = "vae_4layer_v2_latent" + str(latent_dim) + "_lastfilter" + str(last_filter)

    encoder_input = keras.Input(shape=(24,24,1))
    x = keras.layers.Conv2D(num_filters,(4,4),2,activation="relu")(encoder_input)
    x = keras.layers.Conv2D(num_filters,(3,3),1,activation="relu")(x)
    x = keras.layers.Conv2D(num_filters,(3,3),1,activation="relu")(x)
    x = keras.layers.Conv2D(last_filter,(3,3),1,activation="relu")(x)
    x = keras.layers.Flatten()(x)
    z_mean = keras.layers.Dense(latent_dim, name="latent_z_mean")(x)
    z_log_var = layers.Dense(latent_dim, name = "latent_z_log_var")(x)
    z = keras.layers.Lambda(sampling, output_shape=(latent_dim,), name='latent_samp')([z_mean, z_log_var])
    x = keras.layers.Dense(5*5*last_filter,activation="relu")(z)
    x = keras.layers.Reshape((5,5,last_filter))(x)
    x = keras.layers.Conv2DTranspose(last_filter,(3,3),1,activation="relu")(x)
    x = keras.layers.Conv2DTranspose(num_filters,(3,3),1,activation="relu")(x)
    x = keras.layers.Conv2DTranspose(num_filters,(3,3),1,activation="relu")(x)
    decoder_outputs = keras.layers.Conv2DTranspose(1,(4,4),2,activation="linear",name="output")(x) # QUESTION: Can I use RELU here? I think so because everything is non-negative

    vae = keras.Model(inputs=encoder_input,
                       outputs={"output": decoder_outputs,
                                "latent_z_mean": z_mean,
                                "latent_z_log_var": z_log_var},
                       name="vae_4layer_v2")
    print(name)
    return vae, name

def get_5layer_vae(latent_dim,num_filters,last_filter=8):
    '''
    Args:
        - latent_dim (int) : the dimension of the latent space
        - num_filters (int) : the number of filters to use in the first four encoder
                              and last four decoder layers
        - last_filter (int) : the number of filters to use in the last encoder and
                              first decoder layers
                              
    Returns a CNN-VAE with 5 layers in the encoder and 5 layers in the decoder
    and a name for the model.
    '''
    name = "vae_5layer_latent" + str(latent_dim) + "_lastfilter" + str(last_filter)
    encoder_input = keras.Input(shape=(24,24,1))
    x = keras.layers.Conv2D(num_filters,(3,3),1,activation="relu")(encoder_input)
    x = keras.layers.Conv2D(num_filters,(3,3),1,activation="relu")(x)
    x = keras.layers.Conv2D(num_filters,(2,2),2,activation="relu")(x)
    x = keras.layers.Conv2D(num_filters,(3,3),1,activation="relu")(x)
    x = keras.layers.Conv2D(last_filter,(2,2),2,activation="relu")(x)
    x = keras.layers.Flatten()(x)
    z_mean = keras.layers.Dense(latent_dim, name="latent_z_mean")(x)
    z_log_var = layers.Dense(latent_dim, name = "latent_z_log_var")(x)
    z = keras.layers.Lambda(sampling, output_shape=(latent_dim,), name='latent_samp')([z_mean, z_log_var])
    x = keras.layers.Dense(4*4*last_filter,activation="relu")(z)
    x = keras.layers.Reshape((4,4,last_filter))(x)
    x = keras.layers.Conv2DTranspose(last_filter,(2,2),2,activation="relu")(x)
    x = keras.layers.Conv2DTranspose(num_filters,(3,3),1,activation="relu")(x)
    x = keras.layers.Conv2DTranspose(num_filters,(2,2),2,activation="relu")(x)
    x = keras.layers.Conv2DTranspose(num_filters,(3,3),1,activation="relu")(x)
    decoder_outputs = keras.layers.Conv2DTranspose(1,(3,3),1,activation="linear",name="output")(x) # QUESTION: Can I use RELU here? I think so because everything is non-negative

    vae = keras.Model(inputs=encoder_input,
                       outputs={"output": decoder_outputs,
                                "latent_z_mean": z_mean,
                                "latent_z_log_var": z_log_var},
                       name="vae_5layer")
    print(name)
    return vae,name

# Data loading functions

For ease of training with limited resources, the quark, gluon, W, Z, and top data files were chunked into files of 20k events each.
These files are loaded with the following helper functions into the training and validation tf.Data.datasets for training and into
a numpy array and dictionary of numpy arrays for testing of background and signals, respectively.

In [3]:
epsilon = 1e-6
def load_npy_file(path,transform):
    '''
    Loads and transforms the data file.
    '''
    data = np.load(path.numpy().decode("utf-8"))
    data = transform(data)
    return data

def tf_load_npy_file(path,transform):
    '''
    Wrap the loading function as a tf.py_function
    '''
    def wrapper(path):
        return load_npy_file(path,transform)
      
    data = tf.py_function(func=wrapper, inp=[path], Tout=tf.float32)
    data.set_shape([None, 24, 24, 1]) 
    return tf.data.Dataset.from_tensor_slices(data)

def map_to_vae_targets(x):
    '''
    Since our VAE has three outputs, we need to give targets for each output, so we give dummy targets here.
    '''
    return x, {
        "output": x,
        "latent_z_mean": tf.ones((x.shape[0],), dtype=tf.float32),
        "latent_z_log_var": tf.ones((x.shape[0],), dtype=tf.float32)
    }
        
def get_data_transform(file_path,transform,batchsize=32):
    '''
    Creates the tf.data.Dataset objects for training and validation and returns the test data.
    '''
    print("getting data from: ", file_path)
    train_files = glob.glob(file_path + "train/*.npy")
    
    BATCH_SIZE = batchsize
    
    train_ds = (
        tf.data.Dataset.from_tensor_slices(train_files)
        .shuffle(len(train_files))  # Shuffle file order
        .interleave(
            lambda file_path: tf_load_npy_file(file_path,transform),
            cycle_length=8,  # Number of files to read in parallel
            num_parallel_calls=tf.data.AUTOTUNE
        )
        .shuffle(20000)
        .map(map_to_vae_targets, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

    # Load and transform test data
    jet_g_test = np.load(file_path + 'test/jetImages_g_0_cropped.npy')
    jet_q_test = np.load(file_path + 'test/jetImages_q_0_cropped.npy')
    jet_test = transform(np.concatenate((jet_g_test,jet_q_test)))

    # load and transform validation data
    jet_g_val = np.load(file_path + 'val/jetImages_g_1_cropped.npy')
    jet_q_val = np.load(file_path + 'val/jetImages_q_1_cropped.npy')
    jet_val = transform(np.concatenate((jet_g_val,jet_q_val)))
    
    val_ds = tf.data.Dataset.from_tensor_slices(jet_val)
    val_ds = val_ds.map(map_to_vae_targets, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    return {"train":train_ds, "test" : jet_test,"val": val_ds}

def load_signal(signal,num_files,transform):
    '''
    Loads and transforms signal data. Loads [num_files], with 10k events each, for the given signal
    '''
    files = np.sort(glob.glob("../jet_data/signals/jetImages_" + signal + "*.npy"))[0:num_files]
    data = []
    for file in files:
        print(file)
        data.append(transform(np.load(file)))
    return np.concatenate(data)

def get_signal_dict(num_files,transform):
    '''
    Creates the signal data dict with the loaded and transformed data, using [num_files]
    number of files for each signal
    '''
    return {"w":load_signal("w",num_files,transform),"z":load_signal("z",num_files,transform),"t":load_signal("t",num_files,transform)}

# Train log preprocessed VAEs

In [5]:
def run_vae_log(directory_name,vae,name,beta=1e-4,epochs=300):
    '''
    Helper function to train the given VAE with log preprocessed data and
    calculate the ROC curve data.

    Args:
        - directory_name (str) : directory to save the model and plots
        - vae : keras VAE model to train
        - name (str) : VAE name
        - beta : beta hyperparameter, default value of 1e-4 for log data
        - epochs : the default number of epochs to run

    Returns the trained vae and the object returned from tf.callbacks.History from calling
    model.fit()
    '''
    data = get_data_transform("../jet_data/",lambda x: np.log(x+1))
    signals = get_signal_dict(4,lambda x: np.log(x+1))
    
    opt = Adam(learning_rate=1e-3)
    print("beta: ",beta)
    vaebeta = beta
    vae.compile(optimizer=opt,
                            loss={"output": "mse","latent_z_mean": kl_mu_loss,"latent_z_log_var": kl_sigma_loss},
                            loss_weights={"output": 1. - vaebeta,"latent_z_mean": vaebeta,"latent_z_log_var": vaebeta},
                            metrics={"output": "accuracy"})
    
    model_history,roc_data = train_model(directory_name,name,vae,data['train'],data['test'],data['val'],signals,beta,epochs=epochs)
    return vae,model_history

In [ ]:
# Create 14 VAE architectures
models_3layer = [get_3layer_vae(latent,8,last_filter=4) for latent in [3,6]]
models_3layer_v2 = [get_3layer_vae_v2(latent,8,last_filter=4) for latent in [2,3,6]]
models_4layer = [get_4layer_vae(latent,8,last_filter=4) for latent in [3,6]]
models_4layer_v2 = [get_4layer_vae_v2(latent,8,last_filter=4) for latent in [3,6]]
models_5layer = [get_5layer_vae(latent,8,8) for latent in [3,6]]
models_3layer_extra = [get_3layer_vae(latent_dim=int(num_filters/2),num_filters=num_filters,last_filter=int(num_filters/2)) for num_filters in [16,32,64]]

all_models = np.concatenate([models_3layer,models_3layer_v2,,models_4layer,models_4layer_v2,models_5layer,models_3layer_v2_extra])

# Train and obtain the ROC curve data for each model.
name_suffix = "beta1e-4"
for (model,name) in all_models:
    directory_name = f"log_{name}_{name_suffix}"
    print(directory_name)
    model.summary()
    model_result, model_history = run_vae_log(directory_name,model,directory_name,epochs=300)

# Train truncated preprocessed VAEs

In [ ]:
def run_vae_truncated(directory_name,vae,name,beta=1e-2,epochs=300):
    '''
    Helper function to train the given VAE with truncated preprocessed data and
    calculate the ROC curve data.

    Args:
        - directory_name (str) : directory to save the model and plots
        - vae : keras VAE model to train
        - name (str) : VAE name
        - beta : beta hyperparameter, default value of 1e-2 for truncated data
        - epochs : the default number of epochs to run

    Returns the trained vae and tf.callbacks.History object from model.fit()
    '''
    data = get_data_transform("../jet_data/",lambda x: np.clip(x,None,72))
    signals = get_signal_dict(4,lambda x:np.clip(x,None,72))
    
    opt = Adam(learning_rate=1e-3)
    print("beta: ",beta)
    vaebeta = beta
    vae.compile(optimizer=opt,
                            loss={"output": "mse","latent_z_mean": kl_mu_loss,"latent_z_log_var": kl_sigma_loss},
                            loss_weights={"output": 1. - vaebeta,"latent_z_mean": vaebeta,"latent_z_log_var": vaebeta},
                            metrics={"output": "accuracy"})
    
    model_history,roc_data = train_model(directory_name,name,vae,data['train'],data['test'],data['val'],signals,beta,epochs=epochs)
    return vae,model_history

In [ ]:
# Create 13 VAE architectures
models_3layer = [get_3layer_vae(latent,8,last_filter=4) for latent in [3,6]]
models_3layer_v2 = [get_3layer_vae_v2(latent,8,last_filter=4) for latent in [3,6]]
models_4layer = [get_4layer_vae(latent,8,last_filter=4) for latent in [3,6]]
models_4layer_v2 = [get_4layer_vae_v2(latent,8,last_filter=4) for latent in [3,6]]
models_5layer = [get_5layer_vae(latent,8,8) for latent in [3,6]]
models_3layer_extra = [get_3layer_vae(latent_dim=int(num_filters/2),num_filters=num_filters,last_filter=int(num_filters/2)) for num_filters in [16,32,64]]

all_models = np.concatenate([models_3layer,models_3layer_v2,models_3layer_extra,models_4layer,models_4layer_v2,models_5layer])

# Train and obtain the ROC curve data for each model
name_suffix = "beta1e-2"
for (model,name) in all_models:
    directory_name = f"truncated_{name}_{name_suffix}"
    print(directory_name)
    model.summary()
    model_result, model_history = run_vae_truncated(directory_name,model,directory_name,epochs=300)

# Train scaled preprocessed VAEs

In [ ]:
def run_vae_scaled(directory_name,vae,name,beta=1e-6,epochs=300):
    '''
    Helper function to train the given VAE with scaled preprocessed data and
    calculate the ROC curve data.

    Args:
        - directory_name (str) : directory to save the model and plots
        - vae : keras VAE model to train
        - name (str) : VAE name
        - beta : beta hyperparameter, default value of 1e-2 for truncated data
        - epochs : the default number of epochs to run

    Returns the trained vae and tf.callbacks.History object from model.fit()
    '''
    data = get_data_transform("../jet_data/",lambda x: x/512)
    signals = get_signal_dict(4,lambda x: x/512)
    
    opt = Adam(learning_rate=1e-3)
    print("beta: ",beta)
    vaebeta = beta
    vae.compile(optimizer=opt,
                            loss={"output": "mse","latent_z_mean": kl_mu_loss,"latent_z_log_var": kl_sigma_loss},
                            loss_weights={"output": 1. - vaebeta,"latent_z_mean": vaebeta,"latent_z_log_var": vaebeta},
                            metrics={"output": "accuracy"})
    
    model_history,roc_data = train_model(directory_name,name,vae,data['train'],data['test'],data['val'],signals,beta,epochs=epochs)
    return vae,model_history

In [ ]:
# Create 13 VAE architectures
models_3layer = [get_3layer_vae(latent,8,last_filter=4) for latent in [3,6]]
models_3layer_v2 = [get_3layer_vae_v2(latent,8,last_filter=4) for latent in [3,6]]
models_4layer = [get_4layer_vae(latent,8,last_filter=4) for latent in [3,6]]
models_4layer_v2 = [get_4layer_vae_v2(latent,8,last_filter=4) for latent in [3,6]]
models_5layer = [get_5layer_vae(latent,8,8) for latent in [3,6]]
models_3layer_extra = [get_3layer_vae(latent_dim=int(num_filters/2),num_filters=num_filters,last_filter=int(num_filters/2)) for num_filters in [16,32,64]]

all_models = np.concatenate([models_3layer,models_3layer_v2,models_3layer_extra,models_4layer,models_4layer_v2,models_5layer])

# Train and obtain the ROC curve data for each model
name_suffix = "beta1e-6"
for (model,name) in all_models:
    directory_name = f"scaled{name}_{name_suffix}"
    print(directory_name)
    model.summary()
    model_result, model_history = run_vae_scaled(directory_name,model,directory_name,epochs=300)

The following models were chosen as the teachers for the student models based on good overall performance in the low FPR region while remaining small enough that the chopped VAE would be of a reasonable size:

    "log_vae_4layer_small_latent3_lastfilter4_beta1e-4/log_vae_4layer_small_latent3_lastfilter4_beta1e-4.keras"
    "truncated_vae_3layer_latent6_lastfilter4_beta1e-2/truncated_vae_3layer_latent6_lastfilter4_beta1e-2.keras"
    "scaled_vae_3layer_latent6_lastfilter4_beta1e-6/scaled_vae_3layer_latent6_lastfilter4_beta1e-6.keras"